# 06. 고급 RAG (HyDE / RAG-Fusion / smart)
서브쿼리 LLM = **LM Studio 로컬**(`RAG_LLM_PROVIDER=local`, qwen2.5-7b). OpenRouter 아님.
LLM 안 닿으면 **조용히 plain 벡터검색 폴백** → 아래 `_via` 로 실발동 여부 확인.

In [ ]:
import sys,os,asyncio
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
try: sys.stdout.reconfigure(encoding='utf-8')   # Windows 콘솔/nbconvert cp949 방어
except Exception: pass
from dotenv import load_dotenv; load_dotenv()
from app.rag.hyde import hyde_search
from app.rag.rag_fusion import rag_fusion_search
from app.rag._llm import rag_subquery_provider, llm_enabled
print('서브쿼리 provider:', rag_subquery_provider(), '| enabled:', llm_enabled(rag_subquery_provider()))

def show(tag, docs):
    print(tag)
    for d in docs:
        via = d.get('_via', 'plain폴백')   # _via 없으면 LLM 실패->plain 폴백된 것
        score = d.get('_rrf_score') if d.get('_rrf_score') is not None else round(d.get('distance', 0), 3)
        print(f'   {d.get("name"):<20} via={via}  rrf/dist={score}')

show('HyDE  된장찌개:', asyncio.run(hyde_search('된장찌개', k=3)))
show('Fusion 매콤한 돼지고기 볶음 한식:', asyncio.run(rag_fusion_search('매콤한 돼지고기 볶음 한식', k=3)))
print('* via=hyde/rag_fusion => 실발동 / via=plain폴백 => 서브LLM 미동작(LM Studio/키 확인)')